In [ ]:
import boto3
import time
from botocore.exceptions import ClientError

def get_existing_application_url(eb_client, app_name):
    try:
        environments = eb_client.describe_environments(
            ApplicationName=app_name,
            IncludeDeleted=False
        )
        
        for env in environments['Environments']:
            if 'CNAME' in env and env['Status'] == 'Ready':
                url = f"http://{env['CNAME']}"
                print(f"Existing application found! URL: {url}")
                return url
        return None
    except ClientError:
        return None

def create_application(eb_client, app_name):
    try:
        eb_client.create_application(
            ApplicationName=app_name,
            Description='Python application deployed through boto3'
        )
        print(f"Created application: {app_name}")
    except ClientError as e:
        if e.response['Error']['Code'] == 'ApplicationAlreadyExistsException':
            print(f"Application {app_name} already exists")
        else:
            raise e

def upload_to_s3(s3_client, zip_file_path, bucket_name, s3_key, region):
    try:
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    except ClientError as e:
        if e.response['Error']['Code'] != 'BucketAlreadyOwnedByYou':
            raise e

    s3_client.upload_file(zip_file_path, bucket_name, s3_key)
    return f"s3://{bucket_name}/{s3_key}"

def wait_for_application_version(eb_client, app_name, version_label):
    """Wait for application version to be processed"""
    print(f"Waiting for application version {version_label} to be processed...")
    while True:
        response = eb_client.describe_application_versions(
            ApplicationName=app_name,
            VersionLabels=[version_label]
        )
        
        if not response['ApplicationVersions']:
            print(f"Version {version_label} not found")
            return False
            
        status = response['ApplicationVersions'][0]['Status']
        if status == 'PROCESSED':
            print(f"Application version {version_label} is ready")
            return True
        elif status == 'FAILED':
            print(f"Application version {version_label} processing failed")
            return False
            
        print(f"Current version status: {status}. Waiting...")
        time.sleep(10)

def cleanup_resources(eb_client, s3_client, app_name, env_name, bucket_name):
    """Cleanup all resources"""
    try:
        # Terminate environment
        try:
            eb_client.terminate_environment(
                EnvironmentName=env_name
            )
            print(f"Terminated environment: {env_name}")

            # Wait for environment termination
            while True:
                try:
                    env_status = eb_client.describe_environments(
                        EnvironmentNames=[env_name]
                    )['Environments'][0]['Status']
                    if env_status == 'Terminated':
                        break
                    time.sleep(10)
                except IndexError:
                    break
        except ClientError:
            print("Environment already terminated or not found")

        # Delete application
        try:
            eb_client.delete_application(
                ApplicationName=app_name,
                TerminateEnvByForce=True
            )
            print(f"Deleted application: {app_name}")
        except ClientError:
            print("Application already deleted or not found")

        # Delete S3 bucket
        try:
            bucket = boto3.resource('s3').Bucket(bucket_name)
            bucket.objects.all().delete()
            bucket.delete()
            print(f"Deleted S3 bucket: {bucket_name}")
        except ClientError:
            print("Bucket already deleted or not found")

    except Exception as e:
        print(f"Error during cleanup: {e}")

def main():
    # Configuration
    APP_NAME = "Apna Daal"
    ENV_NAME = f"{APP_NAME}-env"
    REGION = "ap-south-1"
    BUCKET_NAME = f"elasticbeanstalk-{REGION}-{APP_NAME.lower()}"
    ZIP_FILE_PATH = "apnefilekaname.zip"
    S3_KEY = f"{APP_NAME}/v1/apnefilekaname.zip"
    VPC_ID = "apna vpc"
    SUBNET_ID = "apna subnet"


    # Initialize AWS clients
    eb_client = boto3.client('elasticbeanstalk', region_name=REGION)
    s3_client = boto3.client('s3', region_name=REGION)
    ec2_client = boto3.client('ec2', region_name=REGION)

    try:
        existing_url = get_existing_application_url(eb_client, APP_NAME)
        if existing_url:
            print("Using existing application.")
            return existing_url

        create_application(eb_client, APP_NAME)

        s3_location = upload_to_s3(s3_client, ZIP_FILE_PATH, BUCKET_NAME, S3_KEY, REGION)
        print(f"Uploaded application to: {s3_location}")

        version_label = f"{APP_NAME}-v1"
        eb_client.create_application_version(
            ApplicationName=APP_NAME,
            VersionLabel=version_label,
            SourceBundle={
                'S3Bucket': BUCKET_NAME,
                'S3Key': S3_KEY
            },
            AutoCreateApplication=False,
            Process=True
        )

        if not wait_for_application_version(eb_client, APP_NAME, version_label):
            raise Exception("Application version processing failed")

        env_response = eb_client.create_environment(
            ApplicationName=APP_NAME,
            EnvironmentName=ENV_NAME,
            SolutionStackName='64bit Amazon Linux 2023 v4.2.0 running Python 3.12',
            OptionSettings=[
                {
                    'Namespace': 'aws:autoscaling:launchconfiguration',
                    'OptionName': 'IamInstanceProfile',
                    'Value': 'Apna IAM Role'
                },
                {
                    'Namespace': 'aws:autoscaling:launchconfiguration',
                    'OptionName': 'EC2KeyName',
                    'Value': 'Apne key ka name'
                },
                {
                    'Namespace': 'aws:ec2:instances',
                    'OptionName': 'InstanceTypes',
                    'Value': 't3.micro'
                },
                {
                    'Namespace': 'aws:elasticbeanstalk:environment',
                    'OptionName': 'EnvironmentType',
                    'Value': 'SingleInstance'
                },
                {
                    'Namespace': 'aws:elasticbeanstalk:environment:proxy',
                    'OptionName': 'ProxyServer',
                    'Value': 'nginx'
                },
                {
                    'Namespace': 'aws:elasticbeanstalk:healthreporting:system',
                    'OptionName': 'SystemType',
                    'Value': 'basic'
                },
                {
                    'Namespace': 'aws:ec2:vpc',
                    'OptionName': 'VPCId',
                    'Value': VPC_ID
                },
                {
                    'Namespace': 'aws:ec2:vpc',
                    'OptionName': 'Subnets',
                    'Value': SUBNET_ID
                }
            ],
            VersionLabel=version_label,
            Tier={
                'Name': 'WebServer',
                'Type': 'Standard',
            }
        )
        
        while True:
            environments = eb_client.describe_environments(
                ApplicationName=APP_NAME,
                EnvironmentNames=[ENV_NAME]
            )['Environments']
            
            if not environments:
                raise Exception("Environment not found")
                
            env_status = environments[0]['Status']
            if env_status == 'Ready':
                url = f"http://{environments[0]['CNAME']}"
                print(f"Environment is ready: {ENV_NAME}")
                print(f"Application URL: {url}")
                return url
            elif env_status in ['Terminated', 'Failed']:
                raise Exception(f"Environment creation failed with status: {env_status}")
            
            print(f"Current status: {env_status}. Waiting...")
            time.sleep(10)

    except Exception as e:
        print(f"An error occurred: {e}")
        print("Starting cleanup process...")
        cleanup_resources(eb_client, s3_client, APP_NAME, ENV_NAME, BUCKET_NAME)
        raise e

if __name__ == "__main__":
    try:
        url = main()
    except Exception as e:
        print(f"Script execution failed: {e}")
        eb_client = boto3.client('elasticbeanstalk', region_name='ap-south-1')
        s3_client = boto3.client('s3', region_name='ap-south-1')
        cleanup_resources(eb_client, s3_client, "Appname", "App-env", 
                        "elasticbeanstalk-ap-south-1-Appname")

IAM Role Beanstalk already exists.
Application MyApp already exists.
Creating application version from S3 bucket.
